<a href="https://colab.research.google.com/github/240215-alt/AI-ML/blob/main/Logistic-regression(gui).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import io

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

from sklearn.datasets import load_iris, load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler

state = {"df": None, "model": None, "encoder": None, "features": []}

# ------------------------------------------------------------------ widgets
source = widgets.ToggleButtons(options=["Upload CSV", "Iris sample", "Breast cancer sample"],
                               description="Data:")
uploader = widgets.FileUpload(accept=".csv", multiple=False, description="Choose CSV")
data_info = widgets.HTML("<i>No data loaded</i>")

target_dd = widgets.Dropdown(description="Target:", options=[], layout=widgets.Layout(width="350px"))
features_sel = widgets.SelectMultiple(description="Features:", options=[], rows=8,
                                      layout=widgets.Layout(width="350px"))
test_size = widgets.FloatSlider(value=0.25, min=0.1, max=0.5, step=0.05, description="Test size:")
c_box = widgets.BoundedFloatText(value=1.0, min=1e-4, max=1e4, step=0.1, description="C:")
iter_box = widgets.BoundedIntText(value=1000, min=50, max=100000, step=50, description="Max iter:")
train_btn = widgets.Button(description="Train model", button_style="primary", icon="play")

out = widgets.Output()
predict_box = widgets.VBox([])
predict_out = widgets.Output()


# ----------------------------------------------------------------- data loading
def set_data(df, label):
    state["df"], state["model"] = df, None
    data_info.value = f"<b>{label}</b>: {df.shape[0]} rows x {df.shape[1]} columns"
    target_dd.options = list(df.columns)
    likely = [c for c in df.columns
              if str(c).lower() in ("target", "label", "class", "species", "outcome", "y")]
    target_dd.value = likely[0] if likely else df.columns[-1]
    refresh_features()
    with out:
        out.clear_output()
        display(df.head(10))


def refresh_features(*_):
    df = state["df"]
    if df is None:
        return
    numeric = [c for c in df.select_dtypes(include=np.number).columns if c != target_dd.value]
    features_sel.options = numeric
    features_sel.value = tuple(numeric)


def load_sample(kind):
    ds = load_iris(as_frame=True) if kind == "iris" else load_breast_cancer(as_frame=True)
    df = ds.frame.copy()
    df["target"] = df["target"].map(dict(enumerate(ds.target_names)))
    set_data(df, f"{kind} sample")


def on_source_change(change):
    if source.value == "Iris sample":
        load_sample("iris")
    elif source.value == "Breast cancer sample":
        load_sample("cancer")


def on_upload(change):
    v = uploader.value
    if not v:
        return
    item = list(v.values())[0] if isinstance(v, dict) else v[0]  # ipywidgets 7 vs 8
    try:
        df = pd.read_csv(io.BytesIO(bytes(item["content"])))
    except Exception as exc:
        with out:
            out.clear_output()
            print("Could not read file:", exc)
        return
    set_data(df, item["name"])


source.observe(on_source_change, names="value")
uploader.observe(on_upload, names="value")
target_dd.observe(refresh_features, names="value")


# -------------------------------------------------------------------- training
def train(_):
    with out:
        out.clear_output()
        df = state["df"]
        if df is None:
            print("Please load a dataset first.")
            return
        target, features = target_dd.value, list(features_sel.value)
        if not target or not features:
            print("Select a target column and at least one feature.")
            return

        data = df[features + [target]].dropna()
        n_classes = data[target].nunique()
        if n_classes < 2:
            print("The target column needs at least 2 distinct classes.")
            return
        if n_classes > 20 or (pd.api.types.is_float_dtype(data[target]) and n_classes > 10):
            print(f"'{target}' has {n_classes} distinct values, so it looks like a continuous "
                  "measurement rather than a class label.\n"
                  "Logistic regression predicts categories. Choose a categorical target "
                  "(e.g. 'target' / species / yes-no) from the Target dropdown.")
            return

        X = data[features].values.astype(float)
        enc = LabelEncoder()
        y = enc.fit_transform(data[target].astype(str))
        classes = list(enc.classes_)

        try:
            X_tr, X_te, y_tr, y_te = train_test_split(
                X, y, test_size=test_size.value, random_state=42, stratify=y)
        except ValueError:
            X_tr, X_te, y_tr, y_te = train_test_split(
                X, y, test_size=test_size.value, random_state=42)

        model = make_pipeline(StandardScaler(),
                              LogisticRegression(C=c_box.value, max_iter=iter_box.value))
        model.fit(X_tr, y_tr)
        state.update(model=model, encoder=enc, features=features)

        y_pred = model.predict(X_te)
        labels = np.arange(len(classes))
        cm = confusion_matrix(y_te, y_pred, labels=labels)
        print("LOGISTIC REGRESSION RESULTS")
        print("=" * 50)
        print(f"Features         : {', '.join(features)}")
        print(f"Classes          : {', '.join(classes)}")
        print(f"Training rows    : {len(X_tr)}    Test rows: {len(X_te)}")
        print(f"Training accuracy: {accuracy_score(y_tr, model.predict(X_tr)):.4f}")
        print(f"Test accuracy    : {accuracy_score(y_te, y_pred):.4f}\n")
        print("Classification report:")
        print(classification_report(y_te, y_pred, labels=labels, target_names=classes,
                                    zero_division=0))

        lr = model.named_steps["logisticregression"]
        coef = lr.coef_
        coef_df = (pd.DataFrame({"coef": coef[0]}, index=features) if coef.shape[0] == 1
                   else pd.DataFrame(coef.T, index=features, columns=classes))
        print("Coefficients (standardized features):")
        display(coef_df.round(4))
        print("Intercept:", np.round(lr.intercept_, 4))

        draw_plots(model, cm, classes, features, X_tr, y_tr, X_te, y_te)

    build_predict_panel()


def draw_plots(model, cm, classes, features, X_tr, y_tr, X_te, y_te):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

    im = ax1.imshow(cm, cmap="Blues")
    ax1.set_xticks(range(len(classes)))
    ax1.set_yticks(range(len(classes)))
    ax1.set_xticklabels(classes, rotation=45, ha="right")
    ax1.set_yticklabels(classes)
    ax1.set_xlabel("Predicted")
    ax1.set_ylabel("Actual")
    ax1.set_title("Confusion matrix (test set)")
    thresh = cm.max() / 2 if cm.max() else 0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax1.text(j, i, cm[i, j], ha="center", va="center",
                     color="white" if cm[i, j] > thresh else "black")
    fig.colorbar(im, ax=ax1, fraction=0.046)

    if len(features) == 2:
        cmap = "coolwarm" if len(classes) == 2 else "viridis"
        X_all = np.vstack([X_tr, X_te])
        pad = 0.1 * (X_all.max(axis=0) - X_all.min(axis=0) + 1e-9)
        (x0, y0), (x1, y1) = X_all.min(axis=0) - pad, X_all.max(axis=0) + pad
        xx, yy = np.meshgrid(np.linspace(x0, x1, 250), np.linspace(y0, y1, 250))
        zz = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
        ax2.contourf(xx, yy, zz, alpha=0.25, cmap=cmap)
        ax2.scatter(X_tr[:, 0], X_tr[:, 1], c=y_tr, edgecolor="k", s=25, cmap=cmap, label="train")
        ax2.scatter(X_te[:, 0], X_te[:, 1], c=y_te, edgecolor="k", s=60, marker="s", cmap=cmap, label="test")
        ax2.set_xlabel(features[0])
        ax2.set_ylabel(features[1])
        ax2.set_title("Decision boundary")
        ax2.legend(fontsize=8)
    else:
        imp = np.abs(model.named_steps["logisticregression"].coef_).mean(axis=0)
        order = np.argsort(imp)
        ax2.barh(np.array(features)[order], imp[order], color="steelblue")
        ax2.set_title("Feature influence (mean |coef|)")
        ax2.tick_params(axis="y", labelsize=8)

    plt.tight_layout()
    plt.show()


# ------------------------------------------------------------------ prediction
def build_predict_panel():
    df, feats = state["df"], state["features"]
    fields = {f: widgets.FloatText(value=round(float(df[f].mean()), 3), description=f[:22],
                                   style={"description_width": "150px"}) for f in feats}
    btn = widgets.Button(description="Predict", button_style="success")

    def predict(_):
        with predict_out:
            predict_out.clear_output()
            x = np.array([[fields[f].value for f in feats]])
            model, enc = state["model"], state["encoder"]
            pred = model.predict(x)[0]
            proba = model.predict_proba(x)[0]
            print(f"Predicted class: {enc.inverse_transform([pred])[0]}\n")
            for cls, p in zip(enc.classes_, proba):
                print(f"  P({cls}) = {p:.3f}")

    btn.on_click(predict)
    predict_out.clear_output()
    predict_box.children = [widgets.HTML("<h4>Predict a new sample</h4>"), *fields.values(), btn, predict_out]


train_btn.on_click(train)

# -------------------------------------------------------------------- layout
display(widgets.VBox([
    widgets.HTML("<h3>Logistic Regression Classifier</h3>"),
    source, uploader, data_info,
    widgets.HBox([target_dd, features_sel]),
    widgets.HBox([test_size, c_box, iter_box]),
    train_btn,
    out,
    predict_box,
]))